# ETL Process for Data Warehouse Project 1

First, import dependencies such as pandas for efficient and easy data processing.

In [111]:
import pandas as pd

Then, read in the first data set `Airline Dataset Updated - v2.csv` which I have renamed to `booking` since each row contains information for a booking between a passenger and the airline.

In [112]:
booking = pd.read_csv('bookings.csv')

booking.head()

,Passenger ID,First Name,Last Name,Gender,Age,Nationality,Airport Name,Airport Country Code,Country Name,Airport Continent,Continents,Departure Date,Arrival Airport,Pilot Name,Flight Status
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,6/28/2022,CXF,Fransisco Hazeldine,On Time
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,12/26/2022,YCO,Marla Parsonage,On Time
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,EU,Europe,1/18/2022,GNB,Rhonda Amber,On Time
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,9/16/2022,YND,Kacie Commucci,Delayed
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2/25/2022,SEE,Ebonee Tree,On Time


Next the data is cleaned of null or invalid data.

In [113]:
# Check for null data.
booking.info()

# Check for invalid airport codes.
invalid_airports = booking[~booking['Arrival Airport'].str.match(r'^[A-Z]{3}$')]
print(f"Number of invalid airport codes: {len(invalid_airports)}")

# Drop invalid rows.
booking.drop(invalid_airports.index, inplace=True)

<class 'pandas.DataFrame'>
RangeIndex: 98619 entries, 0 to 98618
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Passenger ID          98619 non-null  str  
 1   First Name            98619 non-null  str  
 2   Last Name             98619 non-null  str  
 3   Gender                98619 non-null  str  
 4   Age                   98619 non-null  int64
 5   Nationality           98619 non-null  str  
 6   Airport Name          98619 non-null  str  
 7   Airport Country Code  98619 non-null  str  
 8   Country Name          98619 non-null  str  
 9   Airport Continent     98619 non-null  str  
 10  Continents            98619 non-null  str  
 11  Departure Date        98619 non-null  str  
 12  Arrival Airport       98619 non-null  str  
 13  Pilot Name            98619 non-null  str  
 14  Flight Status         98619 non-null  str  
dtypes: int64(1), str(14)
memory usage: 11.3 MB
Number of invalid air

The invalid rows are dropped. Next, the dimension and fact tables will be created, starting with the passenger dimension table.

In [114]:
# Select columns
dim_passenger = booking[['Passenger ID', 'First Name', 'Last Name', 'Gender', 'Age', 'Nationality']].copy()

# Check for duplicate rows
print(dim_passenger.duplicated(subset=['Passenger ID']).sum())

# Rename columns
dim_passenger.rename(columns={"Passenger ID": "PassengerID", "First Name": "FName", "Last Name": "LName"}, inplace=True)

# Write to csv
dim_passenger.to_csv('dockerfiles/dim_passenger.csv', index=False)

dim_passenger

0


,PassengerID,FName,LName,Gender,Age,Nationality
0,ABVWIg,Edithe,Leggis,Female,62,Japan
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua
2,CdUz2g,Darby,Felgate,Male,67,Russia
3,BRS38V,Dominica,Pyle,Female,71,China
4,9kvTLo,Bay,Pencost,Male,21,China
...,...,...,...,...,...,...
98614,hnGQ62,Gareth,Mugford,Male,85,China
98615,2omEzh,Kasey,Benedict,Female,19,Russia
98616,VUPiVG,Darrin,Lucken,Male,65,Indonesia
98617,E47NtS,Gayle,Lievesley,Female,34,China


There appears to be no duplicate passengers since every passenger ID is unique. So the desired columns are easily selected and written to a csv file. 

Next, I will create the airport table which will be extended later with another dataset.

In [115]:
# Select desired columns
dim_airport = booking[['Arrival Airport', 'Airport Name', 'Country Name', 'Continents']].copy()

# Rename columns for interpretability
dim_airport.rename(columns={"Arrival Airport":"IATA", "Airport Name":"Name", "Country Name":"Country", "Continents":"Continent"}, inplace=True)

# Check for duplicates
print(dim_airport.duplicated(subset=['IATA']).sum())

# Drop duplicates
dim_airport.drop_duplicates(subset=['IATA'], inplace=True)

dim_airport


88675


,IATA,Name,Country,Continent
0,CXF,Coldfoot Airport,United States,North America
1,YCO,Kugluktuk Airport,Canada,North America
2,GNB,Grenoble-Isère Airport,France,Europe
3,YND,Ottawa / Gatineau Airport,Canada,North America
4,SEE,Gillespie Field,United States,North America
...,...,...,...,...
68145,HDN,Yampa Valley Airport,United States,North America
71596,HIJ,Hiroshima Airport,Japan,Asia
73571,CXM,Camaxilo Airport,Angola,Africa
78509,ROR,Babelthuap Airport,Palau,Oceania


As explained in the task sheet, `Arrival Airport` is a mislabelling of the IATA code of the departure airport so it was renamed to reflect this. The other columns were also renamed to reflect better naming conventions in relational databases.

Next, any duplicate airports were dropped since they only need to be listed once.

Now, the pilot table will be made.

In [116]:
# Select desired columns.
dim_pilot = booking[['Pilot Name']].copy()

# Check for duplicates
print(dim_pilot.duplicated(subset=['Pilot Name']).sum())

# Drop duplicates
dim_pilot.drop_duplicates(subset=['Pilot Name'], inplace=True)

# Create PilotID and add to original table
dim_pilot.index.name = 'PilotID'
dim_pilot = dim_pilot.reset_index()
booking = booking.merge(dim_pilot[['PilotID', 'Pilot Name']], on='Pilot Name', how='left')

# Separate name columns
dim_pilot["FName"] = dim_pilot['Pilot Name'].str.split(" ").str[:-1].str.join(" ")
dim_pilot["LName"] = dim_pilot['Pilot Name'].str.split(" ").str[-1]
dim_pilot.drop(columns=["Pilot Name"], inplace=True)

# Write to csv
dim_pilot.to_csv("dockerfiles/dim_pilot.csv", index=False)

dim_pilot

13


,PilotID,FName,LName
0,0,Fransisco,Hazeldine
1,1,Marla,Parsonage
2,2,Rhonda,Amber
3,3,Kacie,Commucci
4,4,Ebonee,Tree
...,...,...,...
97675,98614,Pammie,Kingscote
97676,98615,Dorice,Lochran
97677,98616,Gearalt,Main
97678,98617,Judon,Chasle


The assumption is used that any pilots with the same name, are the same pilot; any duplicates are dropped.

Then each pilot is given a `PilotID` which is merged back into the original table so the fact table can be easily created.

Then, `Pilot Name` is divided into `LName` and `FName` to ensure first normal form and the data is written to csv.

Next, the dimension table for flight status is made.

In [117]:
# Find possible values for flight status
possible_values = booking['Flight Status'].unique()

# Make a dimension table for flight status
dim_flight_status = pd.DataFrame({'Flight Status': possible_values})
dim_flight_status.index.name = 'FlightStatusID'
dim_flight_status = dim_flight_status.reset_index()

# Write to csv
dim_flight_status.to_csv("dockerfiles/dim_flight_status.csv", index=False)

# Replace flight status in original table with ID
booking = booking.merge(dim_flight_status, on='Flight Status', how='left')

dim_flight_status

,FlightStatusID,Flight Status
0,0,On Time
1,1,Delayed
2,2,Cancelled


Next, the fact table will be created.

In [118]:
# Take desired columns
fact_booking = booking[['Passenger ID', 'Arrival Airport', 'PilotID', 'FlightStatusID', 'Departure Date']].copy()

# Change all dates to the same format
fact_booking['Departure Date'] = fact_booking['Departure Date'].str.replace('/', '-')

# Create BookingID
fact_booking.index.name = 'BookingID'
fact_booking = fact_booking.reset_index()

# Rename columns for interpretability
fact_booking.rename(columns={"Passenger ID": "PassengerID", "Arrival Airport": "IATA", "Departure Date": "DepartureDate"}, inplace=True)

# Print to csv
fact_booking.to_csv("dockerfiles/fact_booking.csv", index=False)

fact_booking

,BookingID,PassengerID,IATA,PilotID,FlightStatusID,DepartureDate
0,0,ABVWIg,CXF,0,0,6-28-2022
1,1,jkXXAX,YCO,1,0,12-26-2022
2,2,CdUz2g,GNB,2,0,1-18-2022
3,3,BRS38V,YND,3,1,9-16-2022
4,4,9kvTLo,SEE,4,0,2-25-2022
...,...,...,...,...,...,...
97688,97688,hnGQ62,HAA,98614,2,12-11-2022
97689,97689,2omEzh,IVA,98615,2,10-30-2022
97690,97690,VUPiVG,ABC,98616,0,09-10-2022
97691,97691,E47NtS,GGN,98617,2,10-26-2022


Same dates use `/` where as others use `-`, so all were changed to use `-`. Then each record was given a BookingID and the table was written to csv.

Now the `airport` dataset will be imported.

In [119]:
airport = pd.read_csv('airports.csv')

airport.head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN


In [120]:
airport = airport[['type', 'name', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'iso_region', 'municipality', 'iata_code']].copy()

# --- STEP 1: Primary Merge (IATA) ---
# Drop duplicates in airport_info based on IATA before merging!
step1 = dim_airport.merge(
    airport.drop_duplicates(subset=['iata_code']), # <-- THE FIX
    left_on='IATA', 
    right_on='iata_code', 
    how='left', 
    indicator=True
)

# --- STEP 2: Split Successes and Failures ---
matched_on_iata = step1[step1['_merge'] == 'both'].drop(columns=['_merge'])
failed_rows = step1[step1['_merge'] == 'left_only']
clean_failed_rows = failed_rows[dim_airport.columns] 

# --- STEP 3: Backup Merge (Name) ---
# Drop duplicates in airport_info based on Name before merging!
matched_on_name = clean_failed_rows.merge(
    airport.drop_duplicates(subset=['name']), # <-- THE FIX
    left_on='Name', 
    right_on='name', 
    how='left'
)

# --- STEP 4: Stack everything back together ---
final_airport = pd.concat([matched_on_iata, matched_on_name], ignore_index=True)
final_airport = final_airport[['IATA', 'Name', 'type', 'Continent', 'Country', 'iso_region', 'municipality', 'latitude_deg', 'longitude_deg', 'elevation_ft']]
final_airport.rename(columns={"type": "Type", "iso_region": "Region", "municipality": "Municipality", "latitude_deg": "Latitude", "longitude_deg": "Longitude", "elevation_ft": "Elevation"}, inplace=True)

final_airport.to_csv("dockerfiles/dim_airport.csv", index=False)

final_airport.info()
final_airport

<class 'pandas.DataFrame'>
RangeIndex: 9018 entries, 0 to 9017
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   IATA          9018 non-null   str    
 1   Name          9018 non-null   str    
 2   Type          8919 non-null   str    
 3   Continent     9018 non-null   str    
 4   Country       9018 non-null   str    
 5   Region        8919 non-null   str    
 6   Municipality  8553 non-null   str    
 7   Latitude      8919 non-null   float64
 8   Longitude     8919 non-null   float64
 9   Elevation     8604 non-null   float64
dtypes: float64(3), str(7)
memory usage: 704.7 KB


,IATA,Name,Type,Continent,Country,Region,Municipality,Latitude,Longitude,Elevation
0,CXF,Coldfoot Airport,small_airport,North America,United States,US-AK,Coldfoot,67.252197,-150.203995,1042.0
1,YCO,Kugluktuk Airport,small_airport,North America,Canada,CA-NU,Kugluktuk,67.816408,-115.143070,74.0
2,GNB,Grenoble-Isère Airport,medium_airport,Europe,France,FR-ARA,Grenoble,45.362900,5.329370,1302.0
3,YND,Ottawa / Gatineau Airport,medium_airport,North America,Canada,CA-QC,Gatineau,45.521702,-75.563599,211.0
4,SEE,Gillespie Field,small_airport,North America,United States,US-CA,San Diego/El Cajon,32.826199,-116.972000,388.0
...,...,...,...,...,...,...,...,...,...,...
9013,GWY,Galway Airport,closed,Europe,Ireland,IE-G,Galway,53.300201,-8.941590,81.0
9014,YXI,Bonnechere Airport,closed,North America,Canada,CA-ON,Killaloe,45.663101,-77.602798,595.0
9015,RTW,Saratov Central Airport,closed,Europe,Russian Federation,RU-SAR,NaN,51.564999,46.046700,NaN
9016,PFJ,Patreksfjörður Airport,closed,Europe,Iceland,IS-4,Patreksfjörður,65.555801,-23.965000,11.0


The new airport data is merged into the existing airport dimension table using a left join with IATA as the key. A backup join is performed using airport name as the key since not all records in the new dataset contain IATA.

Finally, we take an external data set sourced from 2022 data from World Development Indicators per country
 https://databank.worldbank.org/source/world-development-indicators# with the following columns:
 - Population
 - GDP per capita (current US$)
 - Average precipitation in depth (mm per year)
 - Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)

In [121]:
wdi = pd.read_csv('wdi.csv')
wdi.head()

# Use pivot_table instead of pivot
dim_country = wdi.pivot_table(
    index=['Country Name', 'Country Code'], 
    columns='Series Name',                  
    values='2022 [YR2022]',
    aggfunc='first'
).reset_index()

# Clean up column names
dim_country.columns.name = None

# Replace '..' with empty strings
dim_country.replace('..', '', inplace=True)

,Country Name,Country Code,Average precipitation in depth (mm per year),Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita),GDP (current US$),GDP per capita (current US$),"Population, total"
0,Afghanistan,AFG,327,0.278420956418618,14497243872.1337,357.261152798144,40578842
1,Africa Eastern and Southern,AFE,,0.858131540848484,1228967879390.01,1679.32762166466,731821393
2,Africa Western and Central,AFW,,0.521524499284441,1063649130876.99,2138.47315259913,497387180
3,Albania,ALB,1485,1.84525761573088,19017247013.8821,7756.96188744253,2451636
4,Algeria,DZA,89,4.16005852930563,225581644702.568,4960.30334332888,45477389
...,...,...,...,...,...,...,...
261,West Bank and Gaza,PSE,,,19165500000,3799.95527015163,5043612
262,World,WLD,,4.66731333968557,102251524725531,12798.1658465319,7989545217
263,"Yemen, Rep.",YEM,167,0.287369793942245,,,38222876
264,Zambia,ZMB,1020,0.524548827570452,29163782140.4858,1447.12310138035,20152938


In [122]:
# Create the mapping dictionary
correction_map = {
    'Bahamas, The': 'Bahamas',
    'Bolivia': 'Bolivia, Plurinational State of',
    'British Virgin Islands': 'Virgin Islands, British',
    'Congo, Dem. Rep.': 'Congo, The Democratic Republic of the',
    'Congo, Rep.': 'Congo',
    "Cote d'Ivoire": "Côte d'Ivoire",
    'Egypt, Arab Rep.': 'Egypt',
    'Gambia, The': 'Gambia',
    'Hong Kong SAR, China': 'Hong Kong',
    'Iran, Islamic Rep.': 'Iran, Islamic Republic of',
    "Korea, Dem. People's Rep.": "Korea, Democratic People's Republic of",
    'Korea, Rep.': 'Korea, Republic of',
    'Kyrgyz Republic': 'Kyrgyzstan',
    'Lao PDR': "Lao People's Democratic Republic",
    'Macao SAR, China': 'Macao',
    'Micronesia, Fed. Sts.': 'Micronesia, Federated States of',
    'Moldova': 'Moldova, Republic of',
    'Puerto Rico (US)': 'Puerto Rico',
    'Slovak Republic': 'Slovakia',
    'Somalia, Fed. Rep.': 'Somalia',
    'St. Kitts and Nevis': 'Saint Kitts and Nevis',
    'St. Lucia': 'Saint Lucia',
    'St. Martin (French part)': 'Saint Martin (French part)',
    'St. Vincent and the Grenadines': 'Saint Vincent and the Grenadines',
    'Tanzania': 'Tanzania, United Republic of',
    'Turkiye': 'Turkey',
    'Venezuela, RB': 'Venezuela, Bolivarian Republic of',
    'Virgin Islands (U.S.)': 'Virgin Islands, U.S.',
    'West Bank and Gaza': 'Palestine, State of',
    'Yemen, Rep.': 'Yemen'
}

# Apply the dictionary to the WDI dataset
dim_country['Country Name'] = dim_country['Country Name'].replace(correction_map)

# List of missing countries
missing_countries = {
    'Norfolk Island', 'Montserrat', 'Martinique', 'Cocos (Keeling) Islands', 
    'Anguilla', 'Mayotte', 'Bonaire, Sint Eustatius and Saba', 
    'Falkland Islands (Malvinas)', 'Christmas Island', 'Guadeloupe', 
    'Cook Islands', 'Jersey', 'French Guiana', 'Réunion', 
    'Saint Helena, Ascension and Tristan da Cunha', 
    'United States Minor Outlying Islands', 'Saint Barthélemy', 'Guernsey', 
    'Western Sahara', 'Taiwan, Province of China', 
    'British Indian Ocean Territory', 'Niue', 'Saint Pierre and Miquelon', 
    'Wallis and Futuna'
}

# Convert the set into a temporary pandas DataFrame
missing_df = pd.DataFrame({'Country Name': list(missing_countries)})

# Append the missing countries to the bottom of your main table
dim_country = pd.concat([dim_country, missing_df], ignore_index=True)

# Define the mapping for the new, database friendly names
column_mapping = {
    'Country Name': 'country_name',
    'Average precipitation in depth (mm per year)': 'avg_precipitation_mm',
    'Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)': 'co2_emissions_pc',
    'GDP (current US$)': 'gdp_usd',
    'GDP per capita (current US$)': 'gdp_per_capita_usd',
    'Population, total': 'total_population'
}

# Drop the country code and rename the remaining columns in one chain
dim_country = (dim_country
           .drop(columns=['Country Code', 'GDP (current US$)'])
           .rename(columns=column_mapping))

# View the cleaned headers
print(dim_country.columns.tolist())

# Write to csv
dim_country.to_csv("dockerfiles/dim_country.csv", index=False)

dim_country.info()
# View the result
dim_country.head()

['country_name', 'avg_precipitation_mm', 'co2_emissions_pc', 'gdp_per_capita_usd', 'total_population']
<class 'pandas.DataFrame'>
RangeIndex: 290 entries, 0 to 289
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   country_name          290 non-null    str  
 1   avg_precipitation_mm  266 non-null    str  
 2   co2_emissions_pc      266 non-null    str  
 3   gdp_per_capita_usd    266 non-null    str  
 4   total_population      266 non-null    str  
dtypes: str(5)
memory usage: 11.5 KB


,country_name,avg_precipitation_mm,co2_emissions_pc,gdp_per_capita_usd,total_population
0,Afghanistan,327,0.278420956418618,357.261152798144,40578842
1,Africa Eastern and Southern,,0.858131540848484,1679.32762166466,731821393
2,Africa Western and Central,,0.521524499284441,2138.47315259913,497387180
3,Albania,1485,1.84525761573088,7756.96188744253,2451636
4,Algeria,89,4.16005852930563,4960.30334332888,45477389


There are countries in the external dataset where the country names do not line up, so these are renamed. Additionally, some countries from `booking` do not appear in the new dataset, so they are added in with null values so that foreign key constraints are satisfied when the database is built.